# 06 - PyTorch Fundamentals

Notebook 07 jumps straight into a convolutional network in PyTorch. This notebook is the
missing step before it: **tensors, autograd, `nn.Module`, and the training loop**.

Everything here maps onto notebook 01, where you wrote it all in NumPy by hand. The point
of this notebook is to see which of your hand-written pieces PyTorch replaces, and which it
still expects you to write.

| You wrote by hand (notebook 01) | PyTorch gives you |
|---|---|
| `forward()` | `nn.Module.forward` and layers like `nn.Linear` |
| `backward()` — the chain rule | `loss.backward()` (autograd) |
| `update()` | `optimizer.step()` |
| the epoch loop | **still yours to write** |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

print("torch version:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Tensors

A tensor is a NumPy array that can (a) live on a GPU and (b) remember how it was computed.
Everything else is familiar.

In [ ]:
a = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
b = torch.ones(2, 2)

print("a =\n", a)
print("\na @ b =\n", a @ b)
print("\nshape", a.shape, "| dtype", a.dtype, "| device", a.device)

# Round-trip with NumPy
arr = np.array([1.0, 2.0, 3.0])
t = torch.from_numpy(arr)
print("\nfrom numpy:", t, "-> back to numpy:", t.numpy())

### Three tensor gotchas that cost beginners hours

1. **dtype.** `torch.tensor([1, 2])` is `int64`; layers want `float32`. Use
   `torch.tensor(x, dtype=torch.float32)` or `.float()`.
2. **Shape.** `nn.Linear` expects `(batch_size, features)`. A 1-D tensor of shape `(n,)`
   is not the same as `(n, 1)`. Most "mat1 and mat2 shapes cannot be multiplied" errors
   are this.
3. **Device.** Model and data must be on the same device, or you get a runtime error.

In [ ]:
x = torch.tensor([1, 2, 3])
print("int tensor:", x.dtype, "->", x.float().dtype)

v = torch.randn(5)
print("\nshape (5,)   :", v.shape)
print("shape (5, 1) :", v.reshape(-1, 1).shape, " <- what a layer wants")
print("shape (1, 5) :", v.unsqueeze(0).shape)

## 2. Autograd — the part that replaces your `backward()`

Set `requires_grad=True` and PyTorch records every operation into a graph. Call
`.backward()` and it walks that graph backwards applying the chain rule — the same
derivation you did by hand in notebook 01.

In [ ]:
# A trivial example you can verify with calculus.
x = torch.tensor(3.0, requires_grad=True)
y = x ** 2 + 5 * x + 1          # dy/dx = 2x + 5

y.backward()

print(f"y  = {y.item():.1f}")
print(f"dy/dx at x=3 = {x.grad.item():.1f}")
print(f"by hand: 2*3 + 5 = {2 * 3 + 5}")

In [ ]:
# The same thing that mattered in notebook 01: gradients of a loss w.r.t. weights.
w = torch.randn(3, 1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
X = torch.randn(10, 3)
y_true = torch.randn(10, 1)

y_pred = X @ w + b
loss = ((y_pred - y_true) ** 2).mean()
loss.backward()

print("loss:", round(loss.item(), 4))
print("dL/dw shape:", w.grad.shape, " (matches w:", tuple(w.shape), ")")
print("dL/db:", b.grad)

> **The shape rule from notebook 01 still holds:** every gradient has the same shape as the
> parameter it belongs to. PyTorch enforces it for you now, but the reasoning is unchanged.

### Gradients accumulate — this is the classic bug

PyTorch **adds** new gradients to whatever is already in `.grad`. If you forget to clear
them, epoch 2 trains on the sum of epochs 1 and 2.

In [ ]:
w = torch.tensor([1.0], requires_grad=True)

for step in range(3):
    loss = (w * 2).sum()
    loss.backward()
    print(f"step {step}: w.grad = {w.grad.item()}   <- should be 2.0 every time")

print("\nNow with zeroing:")
w.grad = None
for step in range(3):
    loss = (w * 2).sum()
    loss.backward()
    print(f"step {step}: w.grad = {w.grad.item()}")
    w.grad = None                 # in real code: optimizer.zero_grad()

**That is why `optimizer.zero_grad()` is in every training loop you will ever read.**
Leave it out and your model trains on garbage while reporting no error at all.

## 3. Building a model with `nn.Module`

Notebook 01's `init_params` + `forward` become a class. `nn.Linear(in, out)` holds the
weight matrix and bias, and initialises them sensibly for you.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self, n_input=2, n_hidden=16, n_output=1):
        super().__init__()
        self.fc1 = nn.Linear(n_input, n_hidden)
        self.fc2 = nn.Linear(n_hidden, n_output)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        return self.fc2(x)          # raw logits - no sigmoid here, see the note below

model = SimpleNet()
print(model)
print("\ntrainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

### Why no sigmoid on the output

In notebook 01 the last step was `sigmoid`, and the loss was binary cross-entropy. In
PyTorch you return the raw **logits** and use `nn.BCEWithLogitsLoss`, which applies the
sigmoid internally in a numerically stable way. Applying sigmoid yourself *and* using
`BCEWithLogitsLoss` double-counts it and quietly ruins training.

Equivalent shorthand for simple stacks:

In [ ]:
same_model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)
print(same_model)

## 4. Loss and optimizer

`optimizer.step()` replaces your hand-written `update()`. Adam is the sensible default —
it adapts the step size per parameter, so it is far less sensitive to the learning rate
than the plain gradient descent you wrote in notebook 01.

In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

print("loss function:", criterion)
print("optimizer:", optimizer.__class__.__name__, "| lr =", optimizer.param_groups[0]["lr"])

## 5. The training loop

**Memorise this shape.** Every PyTorch project you will ever read has this loop in it.

```python
for epoch in range(epochs):
    optimizer.zero_grad()          # 1. clear old gradients
    output = model(X)              # 2. forward
    loss = criterion(output, y)    # 3. how wrong
    loss.backward()                # 4. backward (autograd)
    optimizer.step()               # 5. update weights
```

Five lines. Compare it to notebook 01 — it is the same three ideas, with the gradient
maths handed off to autograd.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split

X_np, y_np = make_moons(n_samples=1000, noise=0.20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_np, y_np, test_size=0.25, random_state=42)

# NumPy -> tensors. Note float32 and the (n, 1) target shape.
X_train = torch.tensor(X_tr, dtype=torch.float32)
y_train = torch.tensor(y_tr, dtype=torch.float32).reshape(-1, 1)
X_test = torch.tensor(X_te, dtype=torch.float32)
y_test = torch.tensor(y_te, dtype=torch.float32).reshape(-1, 1)

print("X_train", X_train.shape, "| y_train", y_train.shape)

In [ ]:
model = SimpleNet()
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

history = {"train_loss": [], "test_loss": []}

for epoch in range(500):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train)
    loss = criterion(logits, y_train)
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = criterion(model(X_test), y_test)
        history["train_loss"].append(loss.item())
        history["test_loss"].append(test_loss.item())

    if epoch % 100 == 0:
        print(f"epoch {epoch:4d} | train loss {loss.item():.4f}")

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        preds = (torch.sigmoid(model(X)) > 0.5).float()
    return (preds == y).float().mean().item()

print(f"\ntrain accuracy: {accuracy(model, X_train, y_train):.3f}")
print(f"test accuracy : {accuracy(model, X_test, y_test):.3f}")

In [ ]:
steps = np.arange(len(history["train_loss"])) * 10
plt.figure(figsize=(9, 4))
plt.plot(steps, history["train_loss"], label="train", color="#4C78A8")
plt.plot(steps, history["test_loss"], label="test", color="#F58518")
plt.xlabel("epoch"); plt.ylabel("BCE loss")
plt.title("Same curve as notebook 01 - different amount of code")
plt.legend(); plt.grid(alpha=.3)
plt.show()

### `model.train()` / `model.eval()` and `torch.no_grad()`

Two different things, both easy to forget:

- **`model.train()` vs `model.eval()`** switches the *behaviour* of dropout and batch norm.
  Dropout must be off at evaluation time. Forget `.eval()` and your test scores are noisy
  and pessimistic.
- **`torch.no_grad()`** stops autograd from building a graph. It saves memory and time,
  and has no effect on the numbers.

Use both whenever you evaluate.

## 6. Datasets and DataLoaders

Notebook 01 used the whole training set for every step. Real training uses **mini-batches** —
faster, and the noise in each batch actually helps generalisation.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)

print(f"{len(train_ds)} samples -> {len(train_loader)} batches of 32")

xb, yb = next(iter(train_loader))
print("one batch:", xb.shape, yb.shape)

In [ ]:
# The training loop with batches - the version you will actually use.
model = SimpleNet()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(30):
    model.train()
    epoch_loss = 0.0
    for xb, yb in train_loader:                  # <- the only new line
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    if epoch % 10 == 0:
        print(f"epoch {epoch:3d} | mean batch loss {epoch_loss / len(train_loader):.4f}")

print(f"\ntest accuracy: {accuracy(model, X_test, y_test):.3f}")

## 7. Saving and loading

Save the `state_dict` (the weights), not the whole model object — pickling a model class
breaks as soon as you refactor the code that defines it.

In [ ]:
from pathlib import Path
Path("models").mkdir(exist_ok=True)

torch.save(model.state_dict(), "models/moons_net.pt")

reloaded = SimpleNet()                                    # rebuild the architecture
reloaded.load_state_dict(torch.load("models/moons_net.pt"))
reloaded.eval()

print("reloaded test accuracy:", round(accuracy(reloaded, X_test, y_test), 3))

---
## Exercises

1. Delete `optimizer.zero_grad()` from the training loop and run it. Describe what happens
   to the loss, and why no error is raised.
2. Add a sigmoid to the end of `SimpleNet.forward` while keeping `BCEWithLogitsLoss`.
   Does it still train? Explain what went wrong mathematically.
3. Swap `torch.optim.Adam` for `torch.optim.SGD(model.parameters(), lr=0.01)`. How many
   more epochs does SGD need to reach the same loss?
4. Add a third `nn.Linear` layer. Does test accuracy improve on this dataset? Connect your
   answer to notebook 00, rule 1.
5. Change `batch_size` to 1, then to 750 (the whole set). Time each and compare the loss
   curves. What are you trading?
6. **Stretch:** write your own `nn.Module` subclass for the *NumPy* network of notebook 01
   with no `nn.Linear` — use `nn.Parameter` and raw matrix multiplication. Confirm it
   trains identically.

## Next

Notebook 07 uses everything here on a convolutional network. The training loop will be the
same five lines.